# Ridge vs XGBoost — Hypothesis Testing

Testăm dacă Ridge performează **semnificativ** mai bine decât XGBoost la nivel de cursă.

**Setup:**
- 64 activități totale, split temporal 85/15 → ~55 train, 9 test
- Training: segment-level (o linie = un segment)
- Evaluare: race-level (suma predicțiilor per activitate vs. timp real)

**Teste statistice:**
- Wilcoxon signed-rank (non-parametric, potrivit pentru sample mic)
- Permutation test (fără asumpții distribuționale)
- Bootstrap CI pentru diferența de MAE

## 1. Încarcă datele și reproduce split-ul train/test

In [ ]:
blob = BlobStorageManager(blob_name='segments_training_features.parquet')
df = blob.download_parquet()
print(f'Loaded: {len(df)} segments, {df["activity_id"].nunique()} activities')
print(f'Features: {[c for c in df.columns if c not in DEFAULT_EXCLUDE_COLUMNS and c != SEGMENT_TARGET_COLUMN]}')

In [ ]:
splitter = TemporalSplitter(test_ratio=0.15)
train_df, test_df = splitter.split_train_test(df)

feature_cols = [c for c in df.columns if c not in DEFAULT_EXCLUDE_COLUMNS and c != SEGMENT_TARGET_COLUMN]

X_train = train_df[feature_cols].fillna(0).values
y_train = train_df[SEGMENT_TARGET_COLUMN].values
X_test  = test_df[feature_cols].fillna(0).values

n_test_activities = test_df['activity_id'].nunique()
print(f'Train: {len(train_df)} segments, {train_df["activity_id"].nunique()} activities')
print(f'Test:  {len(test_df)} segments, {n_test_activities} activities')

## 2. Antrenează modelele cu best hyperparams (din training run)

In [ ]:
# Best params din model_comparison.json
ridge_params = RidgeRegressionParams(alpha=100.0)
xgb_params = XGBoostParams(
    learning_rate=0.03,
    max_depth=4,
    min_child_weight=3,
    n_estimators=100,
    subsample=0.8,
)

ridge = ModelFactory.create('ridge', ridge_params)
xgb   = ModelFactory.create('xgboost', xgb_params)

ridge.fit(X_train, y_train)
xgb.fit(X_train, y_train)

print('Models trained.')

## 3. Predicții la nivel de cursă (sumă segmente)

In [ ]:
pred_df = test_df[['activity_id', SEGMENT_TARGET_COLUMN]].copy()
pred_df['ridge_pred'] = ridge.predict(X_test)
pred_df['xgb_pred']   = xgb.predict(X_test)

race = pred_df.groupby('activity_id').sum()
race.columns = ['actual', 'ridge_pred', 'xgb_pred']

race['ridge_error'] = race['ridge_pred'] - race['actual']
race['xgb_error']   = race['xgb_pred']   - race['actual']
race['ridge_abs_error'] = race['ridge_error'].abs()
race['xgb_abs_error']   = race['xgb_error'].abs()

# Convertim în minute pentru lizibilitate
for col in ['actual', 'ridge_pred', 'xgb_pred', 'ridge_error', 'xgb_error', 'ridge_abs_error', 'xgb_abs_error']:
    race[col + '_min'] = race[col] / 60

display_cols = ['actual_min', 'ridge_pred_min', 'xgb_pred_min', 'ridge_error_min', 'xgb_error_min']
print(race[display_cols].round(1).to_string())

## 4. Statistici descriptive

In [ ]:
def summarize(name, errors):
    abs_e = np.abs(errors)
    print(f'{name}:')
    print(f'  MAE:    {abs_e.mean()/60:.1f} min  ({abs_e.mean():.0f}s)')
    print(f'  Median: {np.median(abs_e)/60:.1f} min')
    print(f'  Std:    {abs_e.std()/60:.1f} min')
    print(f'  Max:    {abs_e.max()/60:.1f} min')
    print(f'  MAPE:   {(abs_e / race["actual"]).mean()*100:.1f}%')
    print()

summarize('Ridge', race['ridge_error'].values)
summarize('XGBoost', race['xgb_error'].values)

diff = race['ridge_abs_error'].values - race['xgb_abs_error'].values
print(f'Ridge mai bun pe {(diff < 0).sum()}/{n_test_activities} curse')
print(f'Diferenta medie Ridge-XGB: {diff.mean()/60:.1f} min ({diff.mean():.0f}s)')

## 5. Hypothesis Testing

**H₀:** Nu există diferență sistematică între erorile Ridge și XGBoost  
**H₁:** Ridge are erori semnificativ mai mici (one-sided)

⚠️ **Notă:** Cu doar 9 curse în test set, puterea statistică este limitată. Un test cu α=0.05 necesită o diferență mare pentru a fi semnificativ.

In [ ]:
ridge_abs = race['ridge_abs_error'].values
xgb_abs   = race['xgb_abs_error'].values

# --- Wilcoxon signed-rank test (non-parametric, paired) ---
stat, p_wilcoxon = stats.wilcoxon(ridge_abs, xgb_abs, alternative='less')
print('=== Wilcoxon Signed-Rank Test ===')
print(f'H₁: Ridge AE < XGBoost AE (one-sided)')
print(f'Statistic: {stat:.3f}')
print(f'p-value:   {p_wilcoxon:.4f}')
print(f'Semnificativ (α=0.05): {"DA" if p_wilcoxon < 0.05 else "NU"}')
print()

# --- Paired t-test (pentru referință) ---
t_stat, p_ttest = stats.ttest_rel(ridge_abs, xgb_abs, alternative='less')
print('=== Paired t-test (pentru referință) ===')
print(f'H₁: Ridge MAE < XGBoost MAE (one-sided)')
print(f't-statistic: {t_stat:.3f}')
print(f'p-value:     {p_ttest:.4f}')
print(f'Semnificativ (α=0.05): {"DA" if p_ttest < 0.05 else "NU"}')

In [ ]:
# --- Permutation test ---
np.random.seed(42)
n_permutations = 10_000

observed_diff = ridge_abs.mean() - xgb_abs.mean()  # negative = Ridge better

perm_diffs = []
for _ in range(n_permutations):
    # Randomly swap Ridge/XGB labels for each activity
    swaps = np.random.randint(0, 2, size=n_test_activities)
    a = np.where(swaps, ridge_abs, xgb_abs)
    b = np.where(swaps, xgb_abs, ridge_abs)
    perm_diffs.append(a.mean() - b.mean())

perm_diffs = np.array(perm_diffs)
p_perm = (perm_diffs <= observed_diff).mean()

print('=== Permutation Test ===')
print(f'Diferență observată (Ridge-XGB MAE): {observed_diff/60:.1f} min ({observed_diff:.0f}s)')
print(f'p-value (one-sided): {p_perm:.4f}')
print(f'Semnificativ (α=0.05): {"DA" if p_perm < 0.05 else "NU"}')

In [ ]:
# --- Bootstrap CI pentru diferența de MAE ---
np.random.seed(42)
n_bootstrap = 10_000

boot_diffs = []
for _ in range(n_bootstrap):
    idx = np.random.choice(n_test_activities, size=n_test_activities, replace=True)
    boot_diffs.append(ridge_abs[idx].mean() - xgb_abs[idx].mean())

boot_diffs = np.array(boot_diffs)
ci_low, ci_high = np.percentile(boot_diffs, [2.5, 97.5])

print('=== Bootstrap 95% CI pentru (Ridge MAE - XGBoost MAE) ===')
print(f'Estimat: {observed_diff/60:.1f} min')
print(f'95% CI:  [{ci_low/60:.1f} min, {ci_high/60:.1f} min]')
print(f'CI exclude 0: {"DA — diferență semnificativă" if ci_high < 0 else "NU — diferența nu e certă"}')

## 6. Vizualizare

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Ridge vs XGBoost — Comparare erori la nivel de cursă', fontsize=14, fontweight='bold')

colors = {'Ridge': '#2196F3', 'XGBoost': '#FF5722'}

# --- 1. Eroare absolută per cursă ---
ax = axes[0, 0]
x = np.arange(n_test_activities)
w = 0.35
bars_r = ax.bar(x - w/2, ridge_abs / 60, w, label='Ridge', color=colors['Ridge'], alpha=0.8)
bars_x = ax.bar(x + w/2, xgb_abs / 60, w, label='XGBoost', color=colors['XGBoost'], alpha=0.8)
ax.axhline(ridge_abs.mean() / 60, color=colors['Ridge'], linestyle='--', linewidth=1.5, label=f'Ridge MAE={ridge_abs.mean()/60:.1f}min')
ax.axhline(xgb_abs.mean() / 60, color=colors['XGBoost'], linestyle='--', linewidth=1.5, label=f'XGB MAE={xgb_abs.mean()/60:.1f}min')
ax.set_xlabel('Cursă (test set)')
ax.set_ylabel('Eroare absolută (minute)')
ax.set_title('Eroare absolută per cursă')
ax.legend(fontsize=8)
ax.set_xticks(x)

# --- 2. Pred vs Actual ---
ax = axes[0, 1]
actual_min = race['actual'].values / 60
ax.scatter(actual_min, race['ridge_pred'].values / 60, color=colors['Ridge'], label='Ridge', s=80, zorder=3)
ax.scatter(actual_min, race['xgb_pred'].values / 60, color=colors['XGBoost'], label='XGBoost', marker='s', s=80, zorder=3)
min_v, max_v = actual_min.min() * 0.95, actual_min.max() * 1.05
ax.plot([min_v, max_v], [min_v, max_v], 'k--', linewidth=1, label='Perfectă')
ax.set_xlabel('Timp real (minute)')
ax.set_ylabel('Timp prezis (minute)')
ax.set_title('Predicție vs Timp real')
ax.legend(fontsize=8)

# --- 3. Distribuție erori (signed) ---
ax = axes[1, 0]
ax.hist(race['ridge_error'].values / 60, bins=8, alpha=0.7, color=colors['Ridge'], label='Ridge', edgecolor='white')
ax.hist(race['xgb_error'].values / 60, bins=8, alpha=0.7, color=colors['XGBoost'], label='XGBoost', edgecolor='white')
ax.axvline(0, color='black', linestyle='-', linewidth=1)
ax.axvline(race['ridge_error'].mean() / 60, color=colors['Ridge'], linestyle='--')
ax.axvline(race['xgb_error'].mean() / 60, color=colors['XGBoost'], linestyle='--')
ax.set_xlabel('Eroare (minute)  [negativ = subestimat]')
ax.set_ylabel('Frecvență')
ax.set_title('Distribuție erori (signed)')
ax.legend(fontsize=8)

# --- 4. Bootstrap CI ---
ax = axes[1, 1]
ax.hist(boot_diffs / 60, bins=50, color='steelblue', alpha=0.7, edgecolor='white')
ax.axvline(observed_diff / 60, color='red', linewidth=2, label=f'Observat: {observed_diff/60:.1f} min')
ax.axvline(ci_low / 60, color='orange', linestyle='--', linewidth=1.5, label=f'95% CI: [{ci_low/60:.1f}, {ci_high/60:.1f}] min')
ax.axvline(ci_high / 60, color='orange', linestyle='--', linewidth=1.5)
ax.axvline(0, color='black', linestyle='-', linewidth=1, label='H₀: diferență = 0')
ax.set_xlabel('Ridge MAE − XGBoost MAE (minute)')
ax.set_ylabel('Frecvență bootstrap')
ax.set_title('Bootstrap CI pentru diferența de MAE')
ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('../results/model_comparison_plots.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot salvat în results/model_comparison_plots.png')

## 7. Concluzie

In [ ]:
print('=== SUMAR FINAL ===')
print(f'Test set: {n_test_activities} curse')
print()
print(f'Ridge   MAE: {ridge_abs.mean()/60:.1f} min | Median: {np.median(ridge_abs)/60:.1f} min')
print(f'XGBoost MAE: {xgb_abs.mean()/60:.1f} min | Median: {np.median(xgb_abs)/60:.1f} min')
print()
print(f'Wilcoxon p={p_wilcoxon:.4f} | Permutation p={p_perm:.4f}')
print(f'Bootstrap 95% CI: [{ci_low/60:.1f}, {ci_high/60:.1f}] min')
print()
if p_wilcoxon < 0.05:
    print('✓ Ridge performează semnificativ mai bine (α=0.05)')
elif p_wilcoxon < 0.10:
    print('~ Diferență marginală (p < 0.10) — probabil real, dar testul are putere mică cu 9 curse')
else:
    print('✗ Nu se poate respinge H₀ la α=0.05')
    print('  Notă: cu doar 9 curse, puterea testului este scăzută.')
    print('  Diferența observată poate fi reală, dar nu avem suficiente date pentru certitudine statistică.')